In [221]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import statsmodels
from statsmodels.tsa.seasonal import MSTL
from statsmodels.graphics.tsaplots import plot_acf
import pandas as pd
import os
import glob
import pandas as pd
# testing connection and reading data
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns",None)

In [222]:
HIERARCHY_LEVELS = {
    "total": [],
    "state": ["state_id"],
    "store": ["store_id"],
    "category": ["cat_id"],
    "department": ["dept_id"],
    "state_category": ["state_id", "cat_id"],
    "state_department": ["state_id", "dept_id"],
    "store_category": ["store_id", "cat_id"],
    "store_department": ["store_id", "dept_id"],
    "item": ["item_id"],
    "state_item": ["item_id", "state_id"],
    "item_store": ["item_id", "store_id"],
}


def calculate_hierarchy_scales(train_df: pd.DataFrame):
    results = []

    for level, group_cols in HIERARCHY_LEVELS.items():
        if group_cols:
            daily_sales = (
                train_df.groupby(by=group_cols + ["date"], as_index=False)["sales"]
                .sum()
                .sort_values(by=group_cols + ["date"])
                .reset_index(drop=True)
            )

            daily_sales["cum_sales"] = daily_sales.groupby(group_cols)["sales"].cumsum()
            sales_started = daily_sales["cum_sales"].shift(1) > 0

            daily_sales["previous_sales"] = daily_sales.groupby(group_cols)[
                "sales"
            ].shift(1)

            daily_sales["diff_sq"] = (
                (daily_sales["sales"] - daily_sales["previous_sales"])
                .pow(2)
                .where(sales_started)
            )

            scale_df = (
                daily_sales.groupby(by=group_cols)["diff_sq"]
                .mean()
                .reset_index(name="scale")
            )
            scale_df["series_id"] = scale_df[group_cols].agg("_".join, axis=1)

        else:
            daily_sales = (
                train_df.groupby(by="date", as_index=False)["sales"]
                .sum()
                .sort_values(by="date")
                .reset_index(drop=True)
            )

            daily_sales["diff_sq"] = daily_sales["sales"].diff().pow(2)
            scale_df = pd.DataFrame(
                {"series_id": ["total"], "scale": daily_sales["diff_sq"].mean()}
            )

        scale_df["level"] = level

        results.append(scale_df[["level", "series_id", "scale"]])

    return pd.concat(results, ignore_index=True)


def calculate_hierarchy_mse(val_df: pd.DataFrame):

    results = []

    for level, group_cols in HIERARCHY_LEVELS.items():
        if group_cols:
            daily = val_df.groupby(by=group_cols + ["date"], as_index=False).agg(
                actual=("sales", sum), prediction=("prediction", sum)
            )

            daily["error_sq"] = (daily["actual"] - daily["prediction"]).pow(2)
            mse_df = (
                daily.groupby(by=group_cols)["error_sq"].mean().reset_index(name="mse")
            )
            mse_df["series_id"] = mse_df[group_cols].agg("_".join, axis=1)
        else:
            daily = val_df.groupby(by="date", as_index=False).agg(
                actual=("sales", sum), prediction=("prediction", sum)
            )
            daily["error_sq"] = (daily["actual"] - daily["prediction"]).pow(2)
            mse_df = pd.DataFrame(
                {"series_id": ["total"], "mse": daily["error_sq"].mean()}
            )

        mse_df["level"] = level
        results.append(mse_df[["level", "series_id", "mse"]])

    return pd.concat(results, ignore_index=True)


def calculate_hierarchy_weights(train_df: pd.DataFrame):
    results = []

    weight_df = train_df[
        train_df["date"] > train_df["date"].max() - pd.Timedelta(days=28)
    ]

    weight_df["dollar_sales"] = weight_df["sales"] * weight_df["sell_price"]

    results = []
    for level, group_col in HIERARCHY_LEVELS.items():
        if group_col:
            level_weight = (
                weight_df.groupby(by=group_col, as_index=False)["dollar_sales"]
                .sum()
                .reset_index()
            )
            level_weight["series_id"] = (
                level_weight[group_col].astype(str).agg("_".join, axis=1)
            )
        else:
            level_weight = pd.DataFrame(
                {
                    "series_id": ["total"],
                    "dollar_sales": weight_df["dollar_sales"].sum(),
                }
            )

        level_weight["level"] = level
        results.append(level_weight[["level", "series_id", "dollar_sales"]])

    return pd.concat(results, ignore_index=True)

In [223]:
GOLD_PATH = "../data/gold/features.parquet"

In [224]:
con = duckdb.connect()

df = con.execute(f"""select * from read_parquet('{GOLD_PATH}') where year>=2015 and year<=2016 order by date""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [225]:
df.head()

,item_id,store_id,state_id,cat_id,dept_id,d,date,wm_yr_wk,launch_date,year,month,weekday,sell_price,event_name_1,event_type_1,snap,sales,week_of_year,day_of_week,day_of_month,is_weekend,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12,lag_13,lag_35,lag_14,lag_15,lag_16,lag_17,lag_18,lag_19,lag_20,lag_42,lag_21,lag_22,lag_23,lag_24,lag_25,lag_26,lag_27,lag_49,lag_28,lag_29,lag_30,lag_31,lag_32,lag_33,lag_34,lag_56,rolling_positive_rate_7,rolling_positive_rate_28,rolling_zero_rate_28,days_since_last_sale,rolling_mean_7,rolling_mean_28,rolling_mean_56,rolling_positive_mean_7,rolling_positive_mean_28,rolling_positive_median_28,rolling_positive_std_28,rolling_positive_max_28,price_unique_count,price_max,price_min,price_mean,price_norm,price_weekly_change,price_change_m,price_change_y,price_vs_usual,price_vs_dept,store_dept_rolling_avg,enc_item_mean,enc_item_std,enc_store_mean,enc_store_std,enc_item_store_mean,enc_item_store_std,enc_store_dept_mean,enc_store_dept_std,enc_store_cat_mean,enc_store_cat_std,trend_28_vs_long,trend_7_vs_28,trend_28_vs_56,item_vs_dept,item_vs_cat,item_store_vs_global
0,FOODS_1_175,TX_2,TX,FOODS,FOODS_1,1434,2015-01-01,11448,2013-01-01,2015,1,Thursday,2.98,NewYear,National,1,0,1,4,1,0,0,0,0,0,0,1,2,0,0,5,2,2,2,0,0,0,0,0,0,0,0,0,1,0,1,1,0,4,1,0,1,1,0.71,0.54,0.46,28,1.14,1.14,1.14,1.60,1.60,1.0,1.12,4,1,2.98,2.98,2.98,1.00,1.0,1.0,1.00,1.00,0.88,244.79,0.760455,1.261206,1.485225,4.876509,0.577525,0.967804,1.368553,3.328576,2.183064,6.43267,2.147094,1.0,1.0,0.906066,0.568009,0.759446
1,FOODS_1_141,TX_2,TX,FOODS,FOODS_1,1434,2015-01-01,11448,2013-01-01,2015,1,Thursday,3.54,NewYear,National,1,1,1,4,1,0,0,0,0,0,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0.14,0.21,0.79,34,0.14,0.14,0.14,1.00,1.83,1.5,1.17,4,2,3.54,3.28,3.41,1.00,1.0,1.0,1.00,1.04,1.05,244.79,0.329872,0.752671,1.485225,4.876509,0.243243,0.624072,1.368553,3.328576,2.183064,6.43267,0.986667,1.0,1.0,0.175368,0.109937,0.737387
2,FOODS_1_057,TX_2,TX,FOODS,FOODS_1,1434,2015-01-01,11448,2013-03-09,2015,1,Thursday,2.24,NewYear,National,1,0,1,4,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0.29,0.29,0.71,28,0.29,0.29,0.29,1.00,1.38,1.0,0.74,3,6,2.68,0.98,2.32,0.84,1.0,1.0,1.00,0.97,0.66,244.79,0.323614,0.765568,1.485225,4.876509,0.198113,0.576492,1.368553,3.328576,2.183064,6.43267,1.968571,1.0,1.0,0.284973,0.178648,0.612190
3,FOODS_1_034,TX_2,TX,FOODS,FOODS_1,1434,2015-01-01,11448,2013-01-01,2015,1,Thursday,6.05,NewYear,National,1,3,1,4,1,0,0,2,1,0,1,1,1,2,0,0,0,1,0,0,0,0,0,0,0,2,3,0,1,1,0,0,0,0,1,0,0,0,0.14,0.36,0.64,32,0.14,0.14,0.14,1.00,1.40,1.0,0.70,3,4,6.27,6.05,6.08,0.96,1.0,1.0,0.99,1.00,1.79,244.79,0.299004,0.692366,1.485225,4.876509,0.391181,0.725429,1.368553,3.328576,2.183064,6.43267,0.613527,1.0,1.0,0.175368,0.109937,1.308278
4,FOODS_1_099,TX_2,TX,FOODS,FOODS_1,1434,2015-01-01,11448,2013-03-23,2015,1,Thursday,0.98,NewYear,National,1,6,1,4,1,0,0,7,17,9,7,11,8,7,5,5,4,2,2,5,1,11,0,2,1,4,1,6,1,3,1,5,2,9,3,5,11,3,1.00,1.00,0.00,28,5.14,5.14,5.14,5.14,5.96,6.0,3.08,11,2,0.98,0.94,0.96,1.00,1.0,1.0,1.04,1.02,0.29,244.79,5.055690,5.010721,1.485225,4.876509,6.075563,4.486417,1.368553,3.328576,2.183064,6.43267,0.862472,1.0,1.0,3.828861,2.400296,1.201728


In [226]:
print(df["date"].min(),df["date"].max())

2015-01-01 00:00:00 2016-05-22 00:00:00


In [227]:
train_df = df[df["d"]<=1913]
test_df = df[df["d"]>1913] # contains only features

In [228]:
test_df = pd.read_parquet("../artifacts/test_predictions_ensemble.parquet")

In [229]:
test_df.head()

,item_id,store_id,cat_id,dept_id,state_id,date,sales,sell_price,pred_global,pred_store,prediction
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,2016-04-25,1,2.98,0.839267,0.858988,0.849128
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,2016-04-25,0,1.98,0.163945,0.166827,0.165386
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,2016-04-25,1,2.50,1.082707,1.128969,1.105838
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2016-04-25,3,2.98,0.764992,0.777275,0.771133
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,2016-04-25,11,1.68,13.931995,13.512494,13.722244


In [230]:
test_df["series_id"] = test_df["item_id"].astype(str) + "_" + test_df["store_id"].astype(str)

In [231]:
scale_df = calculate_hierarchy_scales(train_df)
mse_df = calculate_hierarchy_mse(test_df)
weights_df = calculate_hierarchy_weights(train_df)

In [232]:
test_unique_df = test_df[["item_id","store_id","cat_id","dept_id","state_id","series_id"]].drop_duplicates()
len(test_unique_df)

30490

In [251]:
def evaluate(train_df: pd.DataFrame, val_df: pd.DataFrame):
    train_df = train_df.astype({"sales": "float64", "sell_price": "float64"})
    val_df = val_df.astype({"sales": "float64", "prediction": "float64"})

    scale_df = calculate_hierarchy_scales(train_df)
    mse_df = calculate_hierarchy_mse(val_df)
    weights = calculate_hierarchy_weights(train_df)

    metrics = scale_df.merge(mse_df, on=["level", "series_id"], how="inner").merge(
        weights, on=["level", "series_id"], how="inner"
    )

    metrics["rmsse"] = np.where(
        metrics["mse"] > 0, np.sqrt(metrics["mse"] / metrics["scale"]), 0.0
    )

    metrics["weight"] = metrics["dollar_sales"] / metrics.groupby(by="level")[
        "dollar_sales"
    ].transform("sum")

    metrics["weighted_rmsse"] = metrics["rmsse"] * metrics["weight"]
    level_scores = (
        metrics.groupby(by="level")["weighted_rmsse"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_rmsse": "wrmsse_level"})
    )

    wrmsse = level_scores["wrmsse_level"].mean()

    return level_scores, wrmsse


test_df["prediction"] = test_df["pred_global"]
level_scores, wrmsse = evaluate(train_df, test_df)
print(level_scores)

               level  wrmsse_level
0           category      0.522711
1         department      0.551010
2               item      0.858992
3         item_store      0.820637
4              state      0.528193
5     state_category      0.544929
6   state_department      0.573927
7         state_item      0.833756
8              store      0.570777
9     store_category      0.593112
10  store_department      0.634374
11             total      0.522720


In [233]:
merge_df = test_unique_df.merge(scale_df,on="series_id",how="inner")
merge_df = merge_df.merge(mse_df, on="series_id",how="inner")
merge_df = merge_df.merge(weights_df, on="series_id",how="inner")
merge_df.drop(columns=["level_x","level_y","series_id"],inplace=True)
merge_df["rmsse"] = merge_df["mse"]/merge_df["scale"]
merge_df

,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,3.818372,1.572459,item_store,65.559998,0.411814
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,0.590814,0.042797,item_store,0.000000,0.072438
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,3.154489,0.885774,item_store,20.000000,0.280798
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2.282427,1.672022,item_store,119.199997,0.732563
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,47.085595,27.149974,item_store,750.959961,0.576609
...,...,...,...,...,...,...,...,...,...,...
30485,HOBBIES_2_017,CA_4,HOBBIES,HOBBIES_2,CA,0.138593,0.002040,item_store,0.000000,0.014720
30486,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,0.379237,0.162484,item_store,9.700001,0.428449
30487,HOBBIES_2_102,CA_4,HOBBIES,HOBBIES_2,CA,0.130233,0.065299,item_store,29.849998,0.501405
30488,HOBBIES_2_073,CA_4,HOBBIES,HOBBIES_2,CA,0.083146,0.002567,item_store,0.500000,0.030872


In [234]:
merge_df[merge_df["scale"].isna()] 
# 16 series has scale as na or 0 , so ignoring it

,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse
6007,FOODS_2_053,WI_1,FOODS,FOODS_2,WI,NaN,0.006307,item_store,0.0,NaN
10137,FOODS_2_394,TX_3,FOODS,FOODS_2,TX,NaN,0.005109,item_store,0.0,NaN
10433,FOODS_2_129,TX_3,FOODS,FOODS_2,TX,NaN,0.266979,item_store,0.0,NaN
11778,FOODS_3_751,CA_4,FOODS,FOODS_3,CA,NaN,1.293718,item_store,0.0,NaN
23294,HOUSEHOLD_1_356,CA_3,HOUSEHOLD,HOUSEHOLD_1,CA,NaN,0.238640,item_store,0.0,NaN
23952,HOBBIES_1_210,CA_1,HOBBIES,HOBBIES_1,CA,NaN,0.011189,item_store,0.0,NaN
25071,HOUSEHOLD_1_356,CA_4,HOUSEHOLD,HOUSEHOLD_1,CA,NaN,0.006384,item_store,0.0,NaN
25694,HOBBIES_1_335,WI_1,HOBBIES,HOBBIES_1,WI,NaN,3.156219,item_store,0.0,NaN
26136,HOBBIES_1_335,CA_3,HOBBIES,HOBBIES_1,CA,NaN,2.047298,item_store,0.0,NaN
26882,HOBBIES_1_335,CA_2,HOBBIES,HOBBIES_1,CA,NaN,0.091686,item_store,0.0,NaN


In [235]:
merge_df = merge_df.dropna()

In [236]:
merge_df.shape

(30474, 10)

In [237]:
import numpy as np

test_price_df = (
    test_df.groupby(["item_id", "store_id"], observed=True)["sell_price"]
    .agg(mean_price="mean")
    .reset_index()
)

test_price_df["price_band"] = pd.cut(
    test_price_df["mean_price"],
    bins=[0, 1, 2, 5, 10, np.inf],
    labels=["<$1", "$1-2", "$2-5", "$5-10", "$10+"],
    right=False,
)

merge_df = merge_df.merge(test_price_df, on=["item_id","store_id"],how="inner")
merge_df


,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse,mean_price,price_band
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,3.818372,1.572459,item_store,65.559998,0.411814,2.98,$2-5
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,0.590814,0.042797,item_store,0.000000,0.072438,1.98,$1-2
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,3.154489,0.885774,item_store,20.000000,0.280798,2.50,$2-5
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2.282427,1.672022,item_store,119.199997,0.732563,2.98,$2-5
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,47.085595,27.149974,item_store,750.959961,0.576609,1.68,$1-2
...,...,...,...,...,...,...,...,...,...,...,...,...
30469,HOBBIES_2_017,CA_4,HOBBIES,HOBBIES_2,CA,0.138593,0.002040,item_store,0.000000,0.014720,0.50,<$1
30470,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,0.379237,0.162484,item_store,9.700001,0.428449,0.97,<$1
30471,HOBBIES_2_102,CA_4,HOBBIES,HOBBIES_2,CA,0.130233,0.065299,item_store,29.849998,0.501405,5.97,$5-10
30472,HOBBIES_2_073,CA_4,HOBBIES,HOBBIES_2,CA,0.083146,0.002567,item_store,0.500000,0.030872,0.50,<$1


In [238]:
last_train_date = train_df["date"].max()
last28 = train_df[train_df["date"]>= last_train_date-pd.Timedelta(days=28)]

volume_df = last28.groupby(["item_id","store_id"])["sales"].agg(mean_sales="mean").reset_index()
merge_df = merge_df.merge(volume_df, on=["item_id","store_id"], how="inner").round(2)
merge_df

,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse,mean_price,price_band,mean_sales
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,3.82,1.57,item_store,65.559998,0.41,2.98,$2-5,0.76
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,0.59,0.04,item_store,0.000000,0.07,1.98,$1-2,0.00
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,3.15,0.89,item_store,20.000000,0.28,2.50,$2-5,0.31
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2.28,1.67,item_store,119.199997,0.73,2.98,$2-5,1.48
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,47.09,27.15,item_store,750.960022,0.58,1.68,$1-2,15.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,HOBBIES_2_017,CA_4,HOBBIES,HOBBIES_2,CA,0.14,0.00,item_store,0.000000,0.01,0.50,<$1,0.00
30470,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,0.38,0.16,item_store,9.700000,0.43,0.97,<$1,0.38
30471,HOBBIES_2_102,CA_4,HOBBIES,HOBBIES_2,CA,0.13,0.07,item_store,29.850000,0.50,5.97,$5-10,0.17
30472,HOBBIES_2_073,CA_4,HOBBIES,HOBBIES_2,CA,0.08,0.00,item_store,0.500000,0.03,0.50,<$1,0.03


In [239]:
merge_df["volume_decile"] = "D00 no sales"

pos = merge_df["mean_sales"] > 0
ranks = merge_df.loc[pos, "mean_sales"].rank(method="first")

merge_df.loc[pos, "volume_decile"] = (
    pd.qcut(
        ranks,
        10,
        labels=[f"D{d:02d}" for d in range(1, 11)]
    ).astype(str)
)
merge_df

,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse,mean_price,price_band,mean_sales,volume_decile
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,3.82,1.57,item_store,65.559998,0.41,2.98,$2-5,0.76,D06
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,0.59,0.04,item_store,0.000000,0.07,1.98,$1-2,0.00,D00 no sales
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,3.15,0.89,item_store,20.000000,0.28,2.50,$2-5,0.31,D03
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2.28,1.67,item_store,119.199997,0.73,2.98,$2-5,1.48,D08
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,47.09,27.15,item_store,750.960022,0.58,1.68,$1-2,15.90,D10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,HOBBIES_2_017,CA_4,HOBBIES,HOBBIES_2,CA,0.14,0.00,item_store,0.000000,0.01,0.50,<$1,0.00,D00 no sales
30470,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,0.38,0.16,item_store,9.700000,0.43,0.97,<$1,0.38,D04
30471,HOBBIES_2_102,CA_4,HOBBIES,HOBBIES_2,CA,0.13,0.07,item_store,29.850000,0.50,5.97,$5-10,0.17,D02
30472,HOBBIES_2_073,CA_4,HOBBIES,HOBBIES_2,CA,0.08,0.00,item_store,0.500000,0.03,0.50,<$1,0.03,D01


In [240]:
last_train_date = train_df["date"].max()
last91 = train_df[train_df["date"] > last_train_date - pd.Timedelta(days=91)].copy()


intermittency_df = (
    last91.assign(has_sale=last91["sales"].gt(0))
    .groupby(["item_id", "store_id"])["has_sale"]
    .agg(nonzero_rate="mean")
    .reset_index()
)

merge_df = merge_df.merge(intermittency_df, on=["item_id", "store_id"], how="left")
merge_df["sale_frequency"] = pd.cut(merge_df["nonzero_rate"], bins=[0, 0.10, 0.50, 1.000001],
    labels=[
        "Highly intermittent",
        "Moderately intermittent",
        "Low intermittency"
    ], include_lowest=True)
merge_df

,item_id,store_id,cat_id,dept_id,state_id,scale,mse,level,dollar_sales,rmsse,mean_price,price_band,mean_sales,volume_decile,nonzero_rate,sale_frequency
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,3.82,1.57,item_store,65.559998,0.41,2.98,$2-5,0.76,D06,0.681319,Low intermittency
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,0.59,0.04,item_store,0.000000,0.07,1.98,$1-2,0.00,D00 no sales,0.021978,Highly intermittent
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,3.15,0.89,item_store,20.000000,0.28,2.50,$2-5,0.31,D03,0.527473,Low intermittency
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2.28,1.67,item_store,119.199997,0.73,2.98,$2-5,1.48,D08,0.637363,Low intermittency
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,47.09,27.15,item_store,750.960022,0.58,1.68,$1-2,15.90,D10,1.000000,Low intermittency
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30469,HOBBIES_2_017,CA_4,HOBBIES,HOBBIES_2,CA,0.14,0.00,item_store,0.000000,0.01,0.50,<$1,0.00,D00 no sales,0.000000,Highly intermittent
30470,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,0.38,0.16,item_store,9.700000,0.43,0.97,<$1,0.38,D04,0.208791,Moderately intermittent
30471,HOBBIES_2_102,CA_4,HOBBIES,HOBBIES_2,CA,0.13,0.07,item_store,29.850000,0.50,5.97,$5-10,0.17,D02,0.065934,Highly intermittent
30472,HOBBIES_2_073,CA_4,HOBBIES,HOBBIES_2,CA,0.08,0.00,item_store,0.500000,0.03,0.50,<$1,0.03,D01,0.032967,Highly intermittent


In [242]:
def slice_unweighted(df,col):
    g = df.groupby(col, observed=True)["rmsse"]
    tbl = pd.DataFrame({"n_series":g.size(), "mean_rmsse":g.mean(), "median_rmsse":g.median()})
    tbl["share_of_series"] = tbl["n_series"]/tbl["n_series"].sum()
    return tbl.sort_values(by="median_rmsse", ascending=False)

for col in ["store_id", "cat_id", "dept_id", "price_band", "volume_decile", "sale_frequency"]:
    print(f"\n====== RMSSE by {col} ==========\n")
    print(slice_unweighted(merge_df, col))


====== RMSSE by store_id ==========

          n_series  mean_rmsse  median_rmsse  share_of_series
store_id                                                     
CA_2          3048    0.844242         0.570         0.100020
WI_2          3048    0.752618         0.530         0.100020
TX_3          3046    0.749448         0.515         0.099954
CA_3          3047    0.731280         0.510         0.099987
CA_4          3046    0.708601         0.510         0.099954
WI_3          3048    0.730338         0.510         0.100020
CA_1          3048    0.828005         0.500         0.100020
WI_1          3047    0.674365         0.480         0.099987
TX_2          3048    0.649580         0.470         0.100020
TX_1          3048    0.694455         0.460         0.100020

====== RMSSE by cat_id ==========

           n_series  mean_rmsse  median_rmsse  share_of_series
cat_id                                                        
FOODS         14366    0.779132          0.53         0.

### Baseline error in entire series

In [243]:
rmsse = merge_df["rmsse"].dropna()

baseline_summary = pd.DataFrame(
    {
        "n_series": len(rmsse),
        "mean_rmsse": rmsse.mean(),
        "median_rmsse": rmsse.median(),
        "p90_rmsse": rmsse.quantile(0.90),
        "p95_rmsse":rmsse.quantile(0.95),
        "p99_rmsse":rmsse.quantile(0.99),
        "pct_series_rmsse_gt_1": (rmsse>1).mean()*100
    },
    index=["baseline"]
)
baseline_summary

,n_series,mean_rmsse,median_rmsse,p90_rmsse,p95_rmsse,p99_rmsse,pct_series_rmsse_gt_1
baseline,30474,0.736297,0.51,1.27,1.79,4.2627,16.020214


#### Observation
- mean rmsse is 0.736
- 50% of the series has rmsse score lesser than 0.51
- 99% of the series has rmsse score lesser than 4.2627 
- This means that the distribution has long right tail containing some of the series with much higher errors than the normal series.

In [244]:
def slice_errors(df, col):
    g = df.groupby(col, observed=True, dropna=False)

    result = g.agg(
        n_series=("rmsse", "count"),
        mean_rmsse=("rmsse", "mean"),
        median_rmsse=("rmsse", "median"),
        pct_rmse_gt_1=("rmsse", lambda x: (x > 1).mean() * 100),
    ).reset_index()

    result["share_of_series"] = result["n_series"] / result["n_series"].sum()

    return result.sort_values("median_rmsse", ascending=False)


dimensions = [
    "store_id",
    "cat_id",
    "dept_id",
    "price_band",
    "volume_decile",
    "sale_frequency",
]

for col in dimensions:
    if col in merge_df.columns:
        print(f"\n========= Error Analysis by {col}============")
        print(slice_errors(merge_df, col))


========= Error Analysis by store_id============
  store_id  n_series  mean_rmsse  median_rmsse  pct_rmse_gt_1  share_of_series
1     CA_2      3048    0.844242         0.570      20.538058         0.100020
8     WI_2      3048    0.752618         0.530      17.749344         0.100020
6     TX_3      3046    0.749448         0.515      16.579120         0.099954
2     CA_3      3047    0.731280         0.510      14.768625         0.099987
3     CA_4      3046    0.708601         0.510      16.546290         0.099954
9     WI_3      3048    0.730338         0.510      17.191601         0.100020
0     CA_1      3048    0.828005         0.500      14.534121         0.100020
7     WI_1      3047    0.674365         0.480      13.291762         0.099987
5     TX_2      3048    0.649580         0.470      14.271654         0.100020
4     TX_1      3048    0.694455         0.460      14.730971         0.100020

========= Error Analysis by cat_id============
      cat_id  n_series  mean_rmss

### Observation
- The FOODS category accounts for 50% of all series, making it the largest contributor to the overall forecasting errors.
- 17% of FOODS series have an RMSSE greater than 1, indicating that these series are more difficult to forecast than the reference baseline.
- Highly intermittent items have a median RMSSE of 0.51 and a mean RMSSE of 0.86. This gap suggests that while most items are forecast reasonably well, a smaller subset has substantially higher errors. Approximately 22% of these items have an RMSSE greater than 1.


In [248]:
merge_df["weight"] = merge_df["dollar_sales"] / merge_df["dollar_sales"].sum()


def weighted_slice_errors(df, col):
    data = df.loc[df["rmsse"].notna() & df["weight"].notna()].copy()

    data["weighted_error"] = data["weight"] * data["rmsse"]

    result = data.groupby(col, observed=True, dropna=False).agg(
        n_series=("weighted_error", "count"),
        mean_rmsse=("rmsse", "mean"),
        median_rmsse = ("rmsse","median"),
        weighted_error_contribution=("weighted_error", "sum"),
        total_weight=("weight", "sum")
    ).reset_index()

    result["share_of_weighted_error"] = (
        result["weighted_error_contribution"]
        / result["weighted_error_contribution"].sum()
    )
    return result.sort_values("weighted_error_contribution", ascending=False)


dimensions = [
    "store_id",
    "cat_id",
    "dept_id",
    "price_band",
    "volume_decile",
    "sale_frequency",
]

for col in dimensions:
    print(f"\n======== WRMSSE for {col}=========\n")
    print(weighted_slice_errors(merge_df, col))



======== WRMSSE for store_id=========

  store_id  n_series  mean_rmsse  median_rmsse  weighted_error_contribution  \
2     CA_3      3047    0.731280         0.510                     0.120758   
8     WI_2      3048    0.752618         0.530                     0.095776   
1     CA_2      3048    0.844242         0.570                     0.093162   
0     CA_1      3048    0.828005         0.500                     0.081615   
6     TX_3      3046    0.749448         0.515                     0.076071   
5     TX_2      3048    0.649580         0.470                     0.067864   
9     WI_3      3048    0.730338         0.510                     0.067696   
7     WI_1      3047    0.674365         0.480                     0.060920   
4     TX_1      3048    0.694455         0.460                     0.060248   
3     CA_4      3046    0.708601         0.510                     0.046626   

   total_weight  share_of_weighted_error  
2      0.155628                 0.156678  
8   

### Observations
- FOOD category contributes 0.464662 (60%) to the total weighted error. It has the largest impact on the forecasting error.
- HOUSEHOLD items contributes less weighted error as 0.224718 compared to FOOD items, but still has the 29% contribution.
- Frequently selling items have highest contribution in weighted error with ~72% compared to highly intermittent products with only ~1.8%

In [268]:
merge_df[merge_df["rmsse"]>1][["item_id","store_id","rmsse","dollar_sales","weight","weighted_error"]]

,item_id,store_id,rmsse,dollar_sales,weight,weighted_error
11,FOODS_3_423,TX_2,1.12,151.979996,0.000041,0.000045
29,FOODS_3_009,TX_2,1.43,59.279999,0.000016,0.000023
33,FOODS_3_549,TX_2,1.56,32.700001,0.000009,0.000014
35,FOODS_3_543,TX_2,1.06,40.279999,0.000011,0.000011
47,FOODS_3_152,TX_2,1.67,5.760000,0.000002,0.000003
...,...,...,...,...,...,...
30450,HOBBIES_2_043,CA_4,1.07,0.000000,0.000000,0.000000
30453,HOBBIES_2_064,CA_4,1.15,3.880000,0.000001,0.000001
30456,HOBBIES_2_133,CA_4,1.97,11.520000,0.000003,0.000006
30458,HOBBIES_2_128,CA_4,2.03,9.940000,0.000003,0.000005


## WHAT KIND — bias vs variance vs timing

In [282]:
pred_df = test_df[["item_id","store_id","cat_id","dept_id","state_id","date","sales","prediction"]].copy()
pred_df["error"]=pred_df["prediction"]-pred_df["sales"]
pred_df["abs_error"]=pred_df["error"].abs()
pred_df["squared_error"]=pred_df["error"]**2

pred_df["actual_zero"]=pred_df["sales"].eq(0)
pred_df["underforecast"]=pred_df["error"].lt(0)
pred_df["overforecast"]=pred_df["error"].gt(0)
pred_df

,item_id,store_id,cat_id,dept_id,state_id,date,sales,prediction,error,abs_error,squared_error,actual_zero,underforecast,overforecast
0,FOODS_3_112,TX_2,FOODS,FOODS_3,TX,2016-04-25,1,0.839267,-0.160733,0.160733,0.025835,False,True,False
1,FOODS_3_380,TX_2,FOODS,FOODS_3,TX,2016-04-25,0,0.163945,0.163945,0.163945,0.026878,True,False,True
2,FOODS_3_719,TX_2,FOODS,FOODS_3,TX,2016-04-25,1,1.082707,0.082707,0.082707,0.006840,False,False,True
3,FOODS_3_104,TX_2,FOODS,FOODS_3,TX,2016-04-25,3,0.764992,-2.235008,2.235008,4.995261,False,True,False
4,FOODS_3_080,TX_2,FOODS,FOODS_3,TX,2016-04-25,11,13.931995,2.931995,2.931995,8.596593,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
853715,HOBBIES_2_080,CA_4,HOBBIES,HOBBIES_2,CA,2016-05-22,0,0.062451,0.062451,0.062451,0.003900,True,False,True
853716,HOBBIES_2_063,CA_4,HOBBIES,HOBBIES_2,CA,2016-05-22,0,0.249128,0.249128,0.249128,0.062065,True,False,True
853717,HOBBIES_2_074,CA_4,HOBBIES,HOBBIES_2,CA,2016-05-22,0,0.087385,0.087385,0.087385,0.007636,True,False,True
853718,HOBBIES_2_078,CA_4,HOBBIES,HOBBIES_2,CA,2016-05-22,0,0.062285,0.062285,0.062285,0.003879,True,False,True


In [280]:
overall_metrics = pd.DataFrame(
    {
        "n_forecasts": len(pred_df),
        "bias": pred_df["error"].mean(),
        "mae": pred_df["error"].abs().mean(),
        "rmse":np.sqrt(pred_df["squared_error"].mean()),
        "error_std":pred_df["error"].std(),
        "pct_overforecast": pred_df["overforecast"].mean()*100,
        "pct_underforecast":pred_df["underforecast"].mean()*100,
        "pct_exact": pred_df["error"].eq(0).mean()*100,
        "pct_actual_zero": pred_df["actual_zero"].mean()*100
    },
    index=["overall"]
).round(3)
overall_metrics

,n_forecasts,bias,mae,rmse,error_std,pct_overforecast,pct_underforecast,pct_exact,pct_actual_zero
overall,853720,-0.099,1.047,2.162,2.16,65.788,34.212,0.0,54.435


### Observation
- At high level, bias is small with -0.099 and MAE with 1.047.
- MAE, the average mean absolute difference between predicted and actual sales is 1.047 units.
- 65% of the observations are overforecasted compared to 34% with underforecasting but bias is still negative value, meaning under-forecasting errors are larger in magnitude on average and outweigh the positive errors overall.
- But overforecasting and underforecasting predictions partly cancels out each other, so the nodel's overall bias looks small even though individual predictions can still be wrong.

In [ ]:
item_store_metrics = pred_df.groupby(["item_id", "store_id"], observed=True).agg(
    n_dates=("date", "unique"),
    bias=("error", "mean"),
    mae=("abs_error", "mean"),
    rmse=("squared_error", lambda x: np.sqrt(x.mean())),
    error_std=("error", "std"),
    pct_actual_zero=("actual_zero", "mean"),
    pct_underforecast=("underforecast", "mean"),
    pct_overforcast=("overforecast", "mean"),
).reset_index()

for col in ["pct_actual_zero",
    "pct_underforecast",
    "pct_overforecast"]:
    item_store_metrics[col]*=100